In [7]:
from dotenv import load_dotenv
import os
load_dotenv("../ncbi.env")

True

In [10]:
import os, json, httpx, xml.etree.ElementTree as ET

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
AUTH = {"email": os.environ["NCBI_EMAIL"], "api_key": os.environ.get("NCBI_API_KEY", "")}
QUERY = '"breast neoplasms"[mh] AND 2019:2026[dp] AND english[la] AND hasabstract NOT "retracted publication"[pt]'

with httpx.Client(timeout=60) as c:
    r = c.get(f"{BASE}/esearch.fcgi", params={**AUTH, "db": "pubmed", "term": QUERY,
              "retmax": 5000, "sort": "pub_date", "retmode": "json"})
    pmids = sorted(r.json()["esearchresult"]["idlist"], key=int)

    with open("../data/corpus.jsonl", "w") as out:
        for i in range(0, len(pmids), 200):
            r = c.post(f"{BASE}/efetch.fcgi", data={**AUTH, "db": "pubmed",
                       "id": ",".join(pmids[i:i+200]), "retmode": "xml"})
            for art in ET.fromstring(r.content).iter("PubmedArticle"):
                abstract = "\n".join(
                    (f"{a.get('Label')}: " if a.get("Label") else "") + "".join(a.itertext())
                    for a in art.iter("AbstractText"))
                if not abstract:
                    continue
                out.write(json.dumps({
                    "pmid": art.findtext(".//MedlineCitation/PMID"),
                    "title": "".join(art.find(".//ArticleTitle").itertext()),
                    "abstract": abstract,
                    "journal": art.findtext(".//Journal/Title"),
                    "pub_year": art.findtext(".//JournalIssue/PubDate/Year"),
                    "pub_types": [p.text for p in art.iter("PublicationType")],
                    "mesh_terms": [d.text for d in art.iter("DescriptorName")],
                    "doi": art.findtext(".//PubmedData/ArticleIdList/ArticleId[@IdType='doi']"),
                }) + "\n")